In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [3]:
train_path = "../data/train.csv"
test_path = "../data/test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

train_df['SalePrice'] = np.log1p(train_df['SalePrice'])

In [4]:
# Separate numerical and categorical features
num_features = train_df.select_dtypes(include=['int64', 'float64']).columns
cat_features = train_df.select_dtypes(include=['object']).columns

# Fill NaN in numerical features with median
num_imputer = SimpleImputer(strategy="median")
train_df[num_features] = num_imputer.fit_transform(train_df[num_features])

# Fill NaN in categorical features with "Missing"
cat_imputer = SimpleImputer(strategy="constant", fill_value="Missing")
train_df[cat_features] = cat_imputer.fit_transform(train_df[cat_features])

In [5]:
from sklearn.preprocessing import OneHotEncoder

# Apply One-Hot Encoding
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# Transform categorical features
encoded_cat = encoder.fit_transform(train_df[cat_features])

# Convert back to DataFrame with proper column names
encoded_cat_df = pd.DataFrame(encoded_cat, columns=encoder.get_feature_names_out(cat_features))

# Concatenate numerical and encoded categorical features
train_df = pd.concat([train_df[num_features], encoded_cat_df], axis=1)

In [6]:
scaler = StandardScaler()
train_df[num_features] = scaler.fit_transform(train_df[num_features])

In [7]:
import numpy as np

# y: target (log-space if already transformed)
# If NOT transformed earlier, uncomment the next line to apply log1p now:
# train_df['SalePrice'] = np.log1p(train_df['SalePrice'])

y = train_df['SalePrice']              # target (expected to be log1p-transformed)
X = train_df.drop(columns=['SalePrice'])  # features only

In [8]:
from sklearn.model_selection import train_test_split

# Hold-out split: 80% train / 20% validation
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [9]:
from sklearn.linear_model import LinearRegression

# Fit baseline linear regression
linreg = LinearRegression()
linreg.fit(X_train, y_train)

# Predict on validation set
y_pred = linreg.predict(X_valid)

In [10]:
from sklearn.metrics import r2_score
import numpy as np

# --- Metrics in log-space (consistent with training target) ---
# R^2: proportion of variance explained (higher is better, <=1)
r2_log = r2_score(y_valid, y_pred)

print("R^2 (log-space):", r2_log)

R^2 (log-space): 0.9064342643302831


In [11]:
from sklearn.metrics import mean_squared_error
import numpy as np

# RMSE in log-space (lower is better)
rmse_log = np.sqrt(mean_squared_error(y_valid, y_pred))

print("RMSE (log-space):", rmse_log)

RMSE (log-space): 0.3309132611719937


In [12]:
# --- Optional: transform back to original price scale ---
# Only do this if y was log1p-transformed
y_valid_price = np.expm1(y_valid)   # inverse of log1p
y_pred_price  = np.expm1(y_pred)

rmse_price = np.sqrt(mean_squared_error(y_valid_price, y_pred_price))
print("RMSE (original price):", rmse_price)

RMSE (original price): 0.9229496410903423


In [13]:
import numpy as np
from sklearn.metrics import mean_squared_error

# Naive predictor: always predict the mean of y_train (log-space)
y_naive = np.full_like(y_valid, fill_value=y_train.mean())
rmse_naive = np.sqrt(mean_squared_error(y_valid, y_naive))
print("Naive RMSE (log-space):", rmse_naive)

Naive RMSE (log-space): 1.084974720423473


In [19]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
import numpy as np

# Cross-validated negative RMSE in log-space (higher is better, so we take negative and flip sign)
cv_scores = cross_val_score(
    LinearRegression(),
    X_train, y_train,
    scoring="neg_root_mean_squared_error",
    cv=5
)
print("CV RMSE (log-space), mean +/- std:", -cv_scores.mean(), "+/-", cv_scores.std())

CV RMSE (log-space), mean +/- std: 0.4011962760374802 +/- 0.02904079522512558


In [20]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
import numpy as np

# Initialize RandomForest
rf = RandomForestRegressor(n_estimators=200, random_state=42)

# Cross-validation RMSE
# scoring='neg_root_mean_squared_error' gives RMSE
rf_scores = cross_val_score(rf, X_train, y_train, 
                            scoring="neg_root_mean_squared_error", cv=5)

print("RandomForest CV RMSE:", -np.mean(rf_scores))

RandomForest CV RMSE: 0.36373872605503493


In [21]:
from sklearn.ensemble import GradientBoostingRegressor

# Initialize Gradient Boosting
gb = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, 
                               max_depth=3, random_state=42)

# Cross-validation RMSE
gb_scores = cross_val_score(gb, X_train, y_train, 
                            scoring="neg_root_mean_squared_error", cv=5)

print("GradientBoosting CV RMSE:", -np.mean(gb_scores))

GradientBoosting CV RMSE: 0.3281092818922041


In [22]:
from xgboost import XGBRegressor

# Initialize XGBoost
xgb = XGBRegressor(n_estimators=500, learning_rate=0.05, 
                   max_depth=3, random_state=42, n_jobs=-1)

# Cross-validation RMSE
xgb_scores = cross_val_score(xgb, X_train, y_train, 
                             scoring="neg_root_mean_squared_error", cv=5)

print("XGBoost CV RMSE:", -np.mean(xgb_scores))

XGBoost CV RMSE: 0.32591351255685325


In [26]:
print("Baseline LinearRegression:", -np.mean(cv_scores))
print("RandomForest:", -np.mean(rf_scores))
print("GradientBoosting:", -np.mean(gb_scores))
print("XGBoost:", -np.mean(xgb_scores))

Baseline LinearRegression: 0.4011962760374802
RandomForest: 0.36373872605503493
GradientBoosting: 0.3281092818922041
XGBoost: 0.32591351255685325


In [27]:
from sklearn.model_selection import GridSearchCV

# Define parameter grid
param_grid = {
    "n_estimators": [300, 500, 800],   # number of trees
    "max_depth": [3, 4, 5],            # tree depth
    "learning_rate": [0.05, 0.1, 0.2], # shrinkage
    "subsample": [0.8, 1.0],           # sample ratio
    "colsample_bytree": [0.8, 1.0]     # feature ratio
}

# Set up GridSearchCV
grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",  # use RMSE
    cv=5,                                   # 5-fold CV
    verbose=2,
    n_jobs=-1                               # use all CPU cores
)

# Fit GridSearchCV
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV RMSE:", -grid_search.best_score_)

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 800, 'subsample': 0.8}
Best CV RMSE: 0.31337057414395253


In [29]:
# Train final model with best params
final_xgb = XGBRegressor(
    colsample_bytree=0.8,
    learning_rate=0.05,
    max_depth=3,
    n_estimators=800,
    subsample=0.8,
    random_state=42
)

final_xgb.fit(X, y)  # train on all data (not just CV splits)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [40]:
# Separate numerical and categorical features
num_features_test = test_df.select_dtypes(include=['int64', 'float64']).columns
cat_features_test = test_df.select_dtypes(include=['object']).columns

# Fill NaN in numerical features with median
num_imputer = SimpleImputer(strategy="median")
test_df[num_features_test] = num_imputer.fit_transform(test_df[num_features_test])

# Fill NaN in categorical features with "Missing"
cat_imputer = SimpleImputer(strategy="constant", fill_value="Missing")
test_df[cat_features_test] = cat_imputer.fit_transform(test_df[cat_features_test])

In [41]:
from sklearn.preprocessing import OneHotEncoder

# Apply One-Hot Encoding
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# Transform categorical features
encoded_cat = encoder.fit_transform(test_df[cat_features_test])

# Convert back to DataFrame with proper column names
encoded_cat_df = pd.DataFrame(encoded_cat, columns=encoder.get_feature_names_out(cat_features_test))

# Concatenate numerical and encoded categorical features
test_df = pd.concat([test_df[num_features_test], encoded_cat_df], axis=1)

In [43]:
print(test_df.dtypes.value_counts())

float64    292
Name: count, dtype: int64
